# FFTW C2C Cython MPI

In [41]:
#=-----------------------------------------------------------------------=#

In [1]:
%%writefile cc2cp.pyx
#cython: boundscheck=False, wraparound=False, cdivision=True
#cython: initializedcheck=False, language_level=3, infer_types=True
import numpy as np, time as tm
from mpi4py_fft import PFFT, newDistArray
from mpi4py import MPI
def ffp():
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    t0 = tm.time()    # <--- time measurement

    N = 500
    NA = np.array([N, N, N], dtype=int)
    f = PFFT(comm, NA, dtype=np.complex128, backend='pyfftw')
    u = newDistArray(f, False)
    u[:] = np.fromfunction( lambda i, j, k:
           (i*N**2 + j*N  + k + 1)*1E-6, u.shape, dtype=u.dtype )

    u_hat = f.forward(u, normalize=False)

    rs = np.array(0, dtype=np.complex128)
    s = np.array(np.sum(u_hat), dtype=np.complex128)
    comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
                op=MPI.SUM, root=0)
    t1 = tm.time()    # <--- time measurement
    return rank, rs, t1-t0

Writing cc2cp.pyx


In [2]:
%%writefile setup.py
from setuptools import setup
from Cython.Build import cythonize

setup(
    ext_modules = cythonize("cc2cp.pyx")
)

Overwriting setup.py


In [3]:
%%bash
#module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
#source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
#conda activate /scratch/app/anaconda3/2020.11
#conda activate --stack ../envs
#rm cc2cs*.so
#LDFLAGS="-L $(pwd)/lo/lib -l fftw3 -l m"  \
#CFLAGS="-I $(pwd)/lo/include"  \
python setup.py build_ext --inplace

Compiling cc2cp.pyx because it changed.
[1/1] Cythonizing cc2cp.pyx
running build_ext
building 'cc2cp' extension
gcc -pthread -B /prj/ampemi/eduardo.miranda2/envs/compiler_compat -Wl,--sysroot=/ -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /prj/ampemi/eduardo.miranda2/envs/include -fPIC -O2 -isystem /prj/ampemi/eduardo.miranda2/envs/include -fPIC -I/prj/ampemi/eduardo.miranda2/envs/include/python3.9 -c cc2cp.c -o build/temp.linux-x86_64-3.9/cc2cp.o
gcc -pthread -B /prj/ampemi/eduardo.miranda2/envs/compiler_compat -Wl,--sysroot=/ -shared -Wl,-rpath,/prj/ampemi/eduardo.miranda2/envs/lib -Wl,-rpath-link,/prj/ampemi/eduardo.miranda2/envs/lib -L/prj/ampemi/eduardo.miranda2/envs/lib -Wl,-rpath,/prj/ampemi/eduardo.miranda2/envs/lib -Wl,-rpath-link,/prj/ampemi/eduardo.miranda2/envs/lib -L/prj/ampemi/eduardo.miranda2/envs/lib build/temp.linux-x86_64-3.9/cc2cp.o -o build/lib.linux-x86_64-3.9/cc2cp.cpython-39-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-3.

In [4]:
! ls cc2cp*.so

cc2cp.cpython-39-x86_64-linux-gnu.so


In [5]:
import cc2cp
print(cc2cp.__doc__)

None


In [6]:
%%writefile cc2cp_c.py
import cc2cp
r,s,t=cc2cp.ffp()
if r == 0 :
     print('S: {:.2f}    T: {:.4f} s'.format(s, t))

Writing cc2cp_c.py


In [7]:
! mpiexec -n 1 python cc2cp_c.py

S: 125.00+0.00j    T: 35.0288 s


In [8]:
! mpiexec -n 16 python cc2cp_c.py

S: 125.00-0.00j    T: 2.8355 s


In [7]:
! cp cc2cp_c.py cc2cp*.so /scratch${PWD#"/prj"}

In [9]:
%%writefile cc2cp.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name cc2cp       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes
#   NTASKS: 1, 4, 9, 16, 36, 49, 64, 81
echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd
cd /scratch${PWD#"/prj"}/    # maybe wrong, need check
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate --stack ./envs
cd fft
                                              
# Executable
EXEC="python cc2cp_c.py"

# Start
echo '$ srun --mpi=pmi2 -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun --mpi=pmi2 -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting cc2cp.srm


Check

In [8]:
%%bash
cd
cd /scratch${PWD#"/prj"}/
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate --stack ./envs
cd fft
mpiexec -n 24 python cc2cp_c.py

S: 125.00-0.00j    T: 4.7382 s


In [10]:
! sbatch --partition=cpu_dev --ntasks=16 cc2cp.srm

Submitted batch job 870592


In [11]:
! squeue -n cc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            870592    cpu_dev   R  0:00     1   24


In [13]:
! squeue -n cc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [14]:
! cat /scratch${PWD#"/prj"}/slurm-870592.out

- Job ID: 870592
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1333
$ srun --mpi=pmi2 -n 16 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.5181 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 1 of (1, 4, 9, 16, 36, 49, 64, 81)

In [15]:
! sbatch --ntasks=1 cc2cp.srm
! sbatch --ntasks=1 cc2cp.srm
! sbatch --ntasks=1 cc2cp.srm

Submitted batch job 870594
Submitted batch job 870595
Submitted batch job 870596


In [24]:
! cat /scratch${PWD#"/prj"}/slurm-870594.out
! cat /scratch${PWD#"/prj"}/slurm-870595.out
! cat /scratch${PWD#"/prj"}/slurm-870596.out

- Job ID: 870594
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1333
$ srun --mpi=pmi2 -n 1 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 25.0510 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870595
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1334
$ srun --mpi=pmi2 -n 1 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 24.4069 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870596
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1337
$ srun --mpi=pmi2 -n 1 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 24.6439 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 4 of (1, 4, 9, 16, 36, 49, 64, 81)

In [16]:
! sbatch --ntasks=4 cc2cp.srm
! sbatch --ntasks=4 cc2cp.srm
! sbatch --ntasks=4 cc2cp.srm

Submitted batch job 870597
Submitted batch job 870598
Submitted batch job 870599


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-870597.out
! cat /scratch${PWD#"/prj"}/slurm-870598.out
! cat /scratch${PWD#"/prj"}/slurm-870599.out

- Job ID: 870597
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1333
$ srun --mpi=pmi2 -n 4 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 8.1220 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870598
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1334
$ srun --mpi=pmi2 -n 4 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 7.9745 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870599
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1337
$ srun --mpi=pmi2 -n 4 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 7.9503 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 9 of (1, 4, 9, 16, 36, 49, 64, 81)

In [17]:
! sbatch --ntasks=9 cc2cp.srm
! sbatch --ntasks=9 cc2cp.srm
! sbatch --ntasks=9 cc2cp.srm

Submitted batch job 870600
Submitted batch job 870601
Submitted batch job 870602


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-870600.out
! cat /scratch${PWD#"/prj"}/slurm-870601.out
! cat /scratch${PWD#"/prj"}/slurm-870602.out

- Job ID: 870600
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1338
$ srun --mpi=pmi2 -n 9 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 3.8127 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870601
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1335
$ srun --mpi=pmi2 -n 9 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 4.1075 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870602
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1335
$ srun --mpi=pmi2 -n 9 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 3.9834 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 16 of (1, 4, 9, 16, 36, 49, 64, 81)

In [18]:
! sbatch --ntasks=16 cc2cp.srm
! sbatch --ntasks=16 cc2cp.srm
! sbatch --ntasks=16 cc2cp.srm

Submitted batch job 870603
Submitted batch job 870604
Submitted batch job 870605


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-870603.out
! cat /scratch${PWD#"/prj"}/slurm-870604.out
! cat /scratch${PWD#"/prj"}/slurm-870605.out

- Job ID: 870603
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1335
$ srun --mpi=pmi2 -n 16 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.4248 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870604
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1335
$ srun --mpi=pmi2 -n 16 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.4446 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870605
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1335
$ srun --mpi=pmi2 -n 16 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 2.4282 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 36 of (1, 4, 9, 16, 36, 49, 64, 81)

In [19]:
! sbatch --ntasks=36 cc2cp.srm
! sbatch --ntasks=36 cc2cp.srm
! sbatch --ntasks=36 cc2cp.srm

Submitted batch job 870606
Submitted batch job 870607
Submitted batch job 870608


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-870606.out
! cat /scratch${PWD#"/prj"}/slurm-870607.out
! cat /scratch${PWD#"/prj"}/slurm-870608.out

- Job ID: 870606
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1333 sdumont1334
$ srun --mpi=pmi2 -n 36 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 5.7413 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870607
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1293 sdumont1294
$ srun --mpi=pmi2 -n 36 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 5.7109 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870608
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1293 sdumont1294
$ srun --mpi=pmi2 -n 36 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 5.3193 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

### 49 of (1, 4, 9, 16, 36, 49, 64, 81)

In [20]:
! sbatch --ntasks=49 cc2cp.srm
! sbatch --ntasks=49 cc2cp.srm
! sbatch --ntasks=49 cc2cp.srm

Submitted batch job 870610
Submitted batch job 870611
Submitted batch job 870612


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-870610.out
! cat /scratch${PWD#"/prj"}/slurm-870611.out
! cat /scratch${PWD#"/prj"}/slurm-870612.out

- Job ID: 870610
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1302 sdumont1321
$ srun --mpi=pmi2 -n 49 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 7.1821 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870611
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1327 sdumont1338 sdumont1353
$ srun --mpi=pmi2 -n 49 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 7.3510 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870612
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1302 sdumont1321
$ srun --mpi=pmi2 -n 49 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 6.6678 s

### 64 of (1, 4, 9, 16, 36, 49, 64, 81)

In [21]:
! sbatch --ntasks=64 cc2cp.srm
! sbatch --ntasks=64 cc2cp.srm
! sbatch --ntasks=64 cc2cp.srm

Submitted batch job 870613
Submitted batch job 870614
Submitted batch job 870615


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-870613.out
! cat /scratch${PWD#"/prj"}/slurm-870614.out
! cat /scratch${PWD#"/prj"}/slurm-870615.out

- Job ID: 870613
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1327 sdumont1338 sdumont1353
$ srun --mpi=pmi2 -n 64 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 7.0327 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870614
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1302 sdumont1321
$ srun --mpi=pmi2 -n 64 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 6.6843 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870615
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1327 sdumont1338 sdumont1353
$ srun --mpi=pmi2 -n 64 python cc2cp_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 5.9390 s

### 81 of (1, 4, 9, 16, 36, 49, 64, 81)

In [22]:
! sbatch --ntasks=81 cc2cp.srm
! sbatch --ntasks=81 cc2cp.srm
! sbatch --ntasks=81 cc2cp.srm

Submitted batch job 870616
Submitted batch job 870617
Submitted batch job 870618


In [23]:
! squeue -n cc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            870597  cpu_small  PD  0:00     1    4
            870598  cpu_small  PD  0:00     1    4
            870599  cpu_small  PD  0:00     1    4
            870600  cpu_small  PD  0:00     1    9
            870601  cpu_small  PD  0:00     1    9
            870602  cpu_small  PD  0:00     1    9
            870603  cpu_small  PD  0:00     1   16
            870604  cpu_small  PD  0:00     1   16
            870605  cpu_small  PD  0:00     1   16
            870606  cpu_small  PD  0:00     2   36
            870607  cpu_small  PD  0:00     2   36
            870608  cpu_small  PD  0:00     2   36
            870610  cpu_small  PD  0:00     3   49
            870611  cpu_small  PD  0:00     3   49
            870612  cpu_small  PD  0:00     3   49
            870613  cpu_small  PD  0:00     3   64
            870614  cpu_small  PD  0:00     3   64
            870615  cpu_small  PD  0:00     3   64
            870616  cpu_small  

In [26]:
! squeue -u $(whoami) -h -t pending,running -r | wc -l

21


In [1]:
! squeue -n cc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [5]:
! cat /scratch${PWD#"/prj"}/slurm-870616.out
! cat /scratch${PWD#"/prj"}/slurm-870617.out
! cat /scratch${PWD#"/prj"}/slurm-870618.out

- Job ID: 870616
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1302 sdumont1321 sdumont1327
$ srun --mpi=pmi2 -n 81 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 5.3318 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870617
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1302 sdumont1321 sdumont1327
$ srun --mpi=pmi2 -n 81 python cc2cp_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 5.3875 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 870618
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1302 sdumont1321 sdumont1327
$ srun --mpi=pmi2 -n 81 python cc2cp_c.py
-- output ------------------------